# CO1 Java Assignment 1 — Campus Smart-Parking System
**Google Colab execution notebook**

This notebook follows the uploaded CO1 assignment (Tasks 1–12). It creates separate task folders, Java source files, compiler/run logs, and test evidence. Run every cell from top to bottom.

**Important:** Task 1, 2, 4–7, and 9–10 also require written explanations/UML/pseudocode/complexity analysis in the report. The notebook includes concise notes and runnable demonstrations where appropriate. Task 3 and Task 8 preserve the supplied `Complete the Code` requirements and expected outputs.

## 0. Setup
Colab runs Linux commands. Install OpenJDK, create the assignment directory, and check the Java version.

In [ ]:
!apt-get -qq update
!apt-get -qq install -y openjdk-17-jdk
!java -version
!javac -version
import os, subprocess, pathlib, textwrap, json, zipfile, time
from pathlib import Path
ROOT = Path("/content/CSA09_Assignment1")
ROOT.mkdir(parents=True, exist_ok=True)
def write_java(task, filename, source):
    folder = ROOT / task
    folder.mkdir(parents=True, exist_ok=True)
    path = folder / filename
    path.write_text(textwrap.dedent(source).strip() + "\n", encoding="utf-8")
    return path
def run_java(task, filename, main_class=None):
    folder = ROOT / task
    src = folder / filename
    compile_result = subprocess.run(["javac", src.name], cwd=folder, capture_output=True, text=True)
    log = [f"$ javac {src.name}", compile_result.stdout, compile_result.stderr]
    if compile_result.returncode == 0:
        cls = main_class or src.stem
        result = subprocess.run(["java", cls], cwd=folder, capture_output=True, text=True)
        log += [f"$ java {cls}", result.stdout, result.stderr, f"EXIT CODE: {result.returncode}"]
    else:
        result = compile_result
    (folder / (src.stem + "_execution.log")).write_text("\n".join(log), encoding="utf-8")
    print("\n".join(x for x in log if x))
    if result.returncode != 0:
        raise RuntimeError(f"{task}/{filename} failed")
    return result.stdout
print("Workspace:", ROOT)

## Task 1 — Understand the problem and identify objects
**Inputs:** vehicle registration, vehicle category, user category/pass status, entry/exit time, requested slot/floor where applicable.  
**Outputs:** allocated slot, unique ticket, entry/exit timestamps, fee, occupancy and validation messages.

**Candidate classes and responsibilities**
- `Vehicle`: common registration and fee behaviour.
- `TwoWheeler`, `Car`, `ElectricCar`: vehicle-specific rates/capabilities.
- `ParkingSlot`: floor/slot identity and current vehicle.
- `ParkingFacility`: owns floors/slots and allocates/releases slots.
- `ParkingTicket`: ticket number, vehicle, entry time and exit processing.
- `ParkingUser` / pass information: student, staff or visitor eligibility.
- `ParkingService`: coordinates entry, exit and payment.

**Business rules:** a slot holds at most one vehicle; a registration cannot have two active tickets; ticket numbers are unique and never reused; invalid slot indexes are rejected; monthly-pass students/staff pay zero; visitors pay the applicable rate; EV charging is charged separately.

**Assumptions:** fee slabs are illustrative and are stated in the implementation; timestamps use Java time classes; the first suitable free slot is allocated; monthly-pass validity is supplied as a Boolean in the demonstration.

**Alternative (objects vs parallel arrays):** object storage keeps registration, type and entry time together and makes adding attributes local to `Vehicle`; parallel arrays require matching indexes and coordinated updates. A linear search is O(n) in either representation. Object references add per-object overhead but improve maintainability; parallel arrays store fields separately and risk index inconsistency.

## Task 2 — Choose the data representation
- Floor and slot indexes: `int`; registration number: `String`; entry/exit: `LocalDateTime`; duration: `long` minutes; fee: `double` for this educational demo (currency-grade systems should use `BigDecimal`).
- Use a jagged `String[][]` because each floor may contain a different number of slots.
- Avoid integer truncation: `(minutes / 60)` loses partial hours. Use ceiling integer arithmetic `(minutes + 59) / 60`.
- Read interactive input with `Scanner`; deterministic test cases are used in this notebook.

**Alternative representations:** a flat array is convenient only when floor slot counts are uniform or an index mapping is maintained; an `ArrayList<String[]>` supports variable floor lengths similarly to a jagged array. Slot lookup by known floor/slot is O(1); scanning for a free slot is O(total slots).

## Task 3 — Control logic and pseudocode
**Allocate:** for each floor, for each slot: if free, store registration and return floor/slot; if none found, return `null`.  
**Release:** validate floor and slot bounds; if occupied, clear it and return `true`; otherwise return `false`.  
**Fee:** bill every started hour: `hours = (minutes + 59) / 60`; apply the vehicle's hourly rate and any daily cap.

The supplied `ParkingFacility` code is completed below without changing its `main()` method.

In [ ]:
src = write_java("Task03", "ParkingFacility.java", """
public class ParkingFacility {
    private final String[][] slots;
    public ParkingFacility(int[] slotsPerFloor) {
        slots = new String[slotsPerFloor.length][];
        for (int f = 0; f < slotsPerFloor.length; f++) {
            if (slotsPerFloor[f] < 0) throw new IllegalArgumentException("Negative slot count");
            slots[f] = new String[slotsPerFloor[f]];
        }
    }
    public int[] allocate(String vehicleNo) {
        if (vehicleNo == null || vehicleNo.isBlank()) throw new IllegalArgumentException("Invalid vehicle number");
        for (int f = 0; f < slots.length; f++) {
            for (int s = 0; s < slots[f].length; s++) {
                if (slots[f][s] == null) {
                    slots[f][s] = vehicleNo;
                    return new int[]{f, s};
                }
            }
        }
        return null;
    }
    public boolean release(int floor, int slot) {
        if (floor < 0 || floor >= slots.length) return false;
        if (slot < 0 || slot >= slots[floor].length) return false;
        if (slots[floor][slot] == null) return false;
        slots[floor][slot] = null;
        return true;
    }
    public static void main(String[] args) {
        ParkingFacility p = new ParkingFacility(new int[]{2, 1});
        System.out.println(java.util.Arrays.toString(p.allocate("TN01AB1111")));
        System.out.println(java.util.Arrays.toString(p.allocate("TN02CD2222")));
        System.out.println(java.util.Arrays.toString(p.allocate("TN03EF3333")));
        System.out.println(java.util.Arrays.toString(p.allocate("TN04GH4444")));
        System.out.println(p.release(0, 1) + " " + p.release(0, 1) + " " + p.release(5, 0));
        System.out.println(java.util.Arrays.toString(p.allocate("TN04GH4444")));
    }
}
""")
run_java("Task03", "ParkingFacility.java")

**Task 3 complexity:** scanning allocation is O(F×S) in the worst case for F floors and at most S slots per floor; release by known indexes is O(1). The jagged array uses O(total slots) space. A free-slot stack/queue can make allocation/release O(1) on average, at the cost of O(total slots) extra references and maintaining the data structure correctly.

## Task 4 — Classes, objects and constructors
The demonstration uses default, parameterized and overloaded constructors, `this(...)` constructor chaining, and a copy constructor. Objects are created on the heap; local reference variables hold references to them.

In [ ]:
write_java("Task04", "ConstructorDemo.java", """
class ParkingTicket {
    private final int ticketNo;
    private final String regNo;
    private final long entryMinute;
    ParkingTicket() { this(0, "UNKNOWN", 0L); }
    ParkingTicket(int ticketNo, String regNo) { this(ticketNo, regNo, System.currentTimeMillis()); }
    ParkingTicket(int ticketNo, String regNo, long entryMinute) {
        if (ticketNo < 0 || regNo == null || regNo.isBlank()) throw new IllegalArgumentException("Invalid ticket");
        this.ticketNo = ticketNo; this.regNo = regNo; this.entryMinute = entryMinute;
    }
    ParkingTicket(ParkingTicket other) { this(other.ticketNo, other.regNo, other.entryMinute); }
    public String toString() { return ticketNo + " | " + regNo + " | " + entryMinute; }
}
public class ConstructorDemo {
    public static void main(String[] args) {
        ParkingTicket a = new ParkingTicket(101, "TN01AB1234", 1000L);
        ParkingTicket b = new ParkingTicket(a);
        System.out.println(a); System.out.println(b);
    }
}
""")
run_java("Task04", "ConstructorDemo.java")

**Constructor comparison:** constructor validation guarantees an object starts valid; default-constructor-plus-setters can expose a temporarily invalid object; static factories can validate and return named creation variants. All create n objects in O(n) total time and O(n) storage when all references are retained; constructor choice does not change the asymptotic cost.

## Task 5 — static members and object life cycle
A static counter belongs to the class and is shared by all facility instances. Occupancy is instance-specific. Once an exited ticket has no reachable references, it becomes eligible for garbage collection.

In [ ]:
write_java("Task05", "StaticDemo.java", """
class TicketCounter {
    private static int nextTicket = 1;
    static int issue() { return nextTicket++; }
}
class Gate {
    private final String name;
    Gate(String name) { this.name = name; }
    int issueTicket() { return TicketCounter.issue(); }
    String getName() { return name; }
}
public class StaticDemo {
    public static void main(String[] args) {
        Gate north = new Gate("North"), south = new Gate("South");
        System.out.println(north.getName() + " ticket " + north.issueTicket());
        System.out.println(south.getName() + " ticket " + south.issueTicket());
    }
}
""")
run_java("Task05", "StaticDemo.java")

**Alternative:** an instance counter gives each of k facilities its own sequence and can produce duplicate ticket numbers across gates. A shared static counter uses one integer for the class; an instance counter uses k integers. For globally unique tickets, use a shared counter or a database-backed sequence.

## Task 6 — Inheritance hierarchy
`Vehicle` is the superclass; `TwoWheeler` and `Car` are subclasses; `ElectricCar` extends `Car` (multilevel inheritance). `super(...)` initializes superclass state. Constructor order is Vehicle → Car → ElectricCar.

In [ ]:
write_java("Task06", "InheritanceDemo.java", """
class BaseVehicle {
    protected String regNo;
    BaseVehicle(String regNo) { this.regNo = regNo; System.out.println("Vehicle constructor"); }
}
class DemoCar extends BaseVehicle {
    DemoCar(String regNo) { super(regNo); System.out.println("Car constructor"); }
}
class DemoElectricCar extends DemoCar {
    DemoElectricCar(String regNo) { super(regNo); System.out.println("ElectricCar constructor"); }
}
public class InheritanceDemo {
    public static void main(String[] args) { new DemoElectricCar("TN01EV1234"); }
}
""")
run_java("Task06", "InheritanceDemo.java")

**UML (text form):** `Vehicle <|-- TwoWheeler`; `Vehicle <|-- Car`; `Car <|-- ElectricCar`; `ElectricCar ..|> Chargeable`.  
**Composition alternative:** `Vehicle` has a `VehicleType`/pricing strategy. Adding a type can mean adding a strategy rather than a subclass. Fee computation remains O(1); inheritance may add a class per type, while composition stores a strategy reference per vehicle.

## Task 7 — Override and specialise behaviour
Subclasses override `ratePerHour()`; `@Override` lets the compiler verify the method signature. `final` is appropriate for immutable registration numbers and constants. `equals()` and `hashCode()` must use the same identity rule.

In [ ]:
write_java("Task07", "OverrideDemo.java", """
abstract class RateVehicle {
    private final String regNo;
    RateVehicle(String regNo) { this.regNo = regNo; }
    abstract double ratePerHour();
    @Override public boolean equals(Object o) {
        return o instanceof RateVehicle && regNo.equals(((RateVehicle)o).regNo);
    }
    @Override public int hashCode() { return regNo.hashCode(); }
}
class RateCar extends RateVehicle {
    RateCar(String regNo) { super(regNo); }
    @Override double ratePerHour() { return 30.0; }
}
public class OverrideDemo {
    public static void main(String[] args) {
        RateCar a = new RateCar("TN01AB1234"), b = new RateCar("TN01AB1234");
        System.out.println(a.ratePerHour());
        System.out.println("Equal registration: " + a.equals(b));
    }
}
""")
run_java("Task07", "OverrideDemo.java")

**Switch alternative:** a single fee method can switch on vehicle type. Each call checks up to k categories (O(k) worst case); adding a category changes the central method. Dynamic dispatch is typically O(1) selection and localises changes to the new subclass.

## Task 8 — Polymorphism (completed supplied FeeDemo)
This cell completes the supplied `Chargeable`, `Vehicle`, `TwoWheeler`, `Car`, `ElectricCar` and `FeeDemo` structure. It demonstrates overloading (`calculateFee(int)` and `calculateFee(LocalTime, LocalTime)`) and runtime dispatch through a `Vehicle[]`.

In [ ]:
write_java("Task08", "FeeDemo.java", """
import java.time.Duration;
import java.time.LocalTime;
interface Chargeable { double chargingFee(int kWh); }
abstract class Vehicle {
    private final String regNo;
    protected Vehicle(String regNo) { this.regNo = regNo; }
    public String getRegNo() { return regNo; }
    protected abstract double ratePerHour();
    public double calculateFee(int minutes) {
        if (minutes < 0) throw new IllegalArgumentException("Minutes cannot be negative");
        int hours = (minutes + 59) / 60;
        return hours * ratePerHour();
    }
    public double calculateFee(LocalTime in, LocalTime out) {
        long minutes = Duration.between(in, out).toMinutes();
        if (minutes < 0) minutes += 24 * 60;
        return calculateFee((int) minutes);
    }
    @Override public boolean equals(Object o) {
        return o instanceof Vehicle && regNo.equals(((Vehicle) o).regNo);
    }
    @Override public int hashCode() { return regNo.hashCode(); }
}
class TwoWheeler extends Vehicle {
    TwoWheeler(String regNo) { super(regNo); }
    @Override protected double ratePerHour() { return 10; }
}
class Car extends Vehicle {
    Car(String regNo) { super(regNo); }
    @Override protected double ratePerHour() { return 30; }
}
class ElectricCar extends Car implements Chargeable {
    ElectricCar(String regNo) { super(regNo); }
    @Override public double chargingFee(int kWh) {
        if (kWh < 0) throw new IllegalArgumentException("kWh cannot be negative");
        return kWh * 12;
    }
}
public class FeeDemo {
    public static void main(String[] args) {
        Vehicle[] parked = { new TwoWheeler("TN01A1"), new Car("TN02B2"),
                             new ElectricCar("TN03C3") };
        for (Vehicle v : parked) {
            double fee = v.calculateFee(130);
            if (v instanceof Chargeable c) fee += c.chargingFee(10);
            System.out.println(v.getClass().getSimpleName() + " " + v.getRegNo()
                    + " -> Rs " + fee);
        }
        System.out.println(new Car("TN02B2").equals(new ElectricCar("TN02B2")));
    }
}
""")
run_java("Task08", "FeeDemo.java")

**Expected output:** TwoWheeler Rs 30.0; Car Rs 90.0; ElectricCar Rs 210.0; final equality result `true`. Equality is based only on registration number in the supplied design; this is a deliberate policy choice but may be too broad if vehicle category is part of identity.

**Dispatch trace:** declared reference type `Vehicle`; actual object type `ElectricCar`; executed override `ratePerHour()` inherited from `Car`, then the `Chargeable` capability is called.

## Task 9 — Abstraction and interfaces
`Vehicle` is abstract because it has shared state (`regNo`) and common implementation while leaving rate selection abstract. `Chargeable` is an interface because charging is a capability that only some vehicles support. A class can implement multiple interfaces.

In [ ]:
write_java("Task09", "AbstractionDemo.java", """
interface Payable { double paymentAmount(); }
abstract class AbstractVehicle {
    private final String regNo;
    AbstractVehicle(String regNo) { this.regNo = regNo; }
    public String getRegNo() { return regNo; }
    abstract double ratePerHour();
}
class EV extends AbstractVehicle implements Payable {
    EV(String regNo) { super(regNo); }
    @Override double ratePerHour() { return 30; }
    @Override public double paymentAmount() { return ratePerHour() * 2; }
}
public class AbstractionDemo {
    public static void main(String[] args) {
        EV ev = new EV("TN01EV0001");
        System.out.println(ev.getRegNo() + " payment Rs " + ev.paymentAmount());
    }
}
""")
run_java("Task09", "AbstractionDemo.java")

**Alternative:** converting `Vehicle` to an interface makes instance fields and constructors unavailable; state must move into implementing classes or a composed object. An abstract class cannot be used as a second superclass, whereas interfaces permit multiple capabilities. Runtime call cost is effectively O(1) in either design; per-object state depends on the implementation.

## Task 10 — Packages and encapsulation
The example uses a package declaration, private fields, a public constructor/getter, and validation. In a larger implementation, split `parking.model`, `parking.service`, and `parking.app` into separate source files.

In [ ]:
pkg = ROOT / "Task10" / "parking" / "model"
pkg.mkdir(parents=True, exist_ok=True)
(pkg / "ParkingSlot.java").write_text("""
package parking.model;
public class ParkingSlot {
    private final int floor;
    private final int slotNumber;
    private String regNo;
    public ParkingSlot(int floor, int slotNumber) {
        if (floor < 0 || slotNumber < 0) throw new IllegalArgumentException("Invalid slot");
        this.floor = floor; this.slotNumber = slotNumber;
    }
    public boolean occupy(String regNo) {
        if (regNo == null || regNo.isBlank() || this.regNo != null) return false;
        this.regNo = regNo; return true;
    }
    public boolean release() {
        if (regNo == null) return false;
        regNo = null; return true;
    }
    public String getRegNo() { return regNo; }
    public String toString() { return "Floor " + floor + ", Slot " + slotNumber + ", Vehicle " + regNo; }
}
""".strip() + "\\n", encoding="utf-8")
app = ROOT / "Task10" / "ParkingSlotApp.java"
app.write_text("""
import parking.model.ParkingSlot;
public class ParkingSlotApp {
    public static void main(String[] args) {
        ParkingSlot slot = new ParkingSlot(0, 5);
        System.out.println(slot.occupy("TN01AB1234"));
        System.out.println(slot);
        System.out.println(slot.release());
    }
}
""".strip() + "\\n", encoding="utf-8")
compile_result = subprocess.run(["javac", "-d", ".", "parking/model/ParkingSlot.java", "ParkingSlotApp.java"], cwd=ROOT/"Task10", capture_output=True, text=True)
print(compile_result.stdout, compile_result.stderr)
if compile_result.returncode: raise RuntimeError("Task10 compile failed")
result = subprocess.run(["java", "ParkingSlotApp"], cwd=ROOT/"Task10", capture_output=True, text=True)
print(result.stdout)
(ROOT/"Task10"/"Task10_execution.log").write_text(result.stdout, encoding="utf-8")

**Wrapper/autoboxing note:** `Integer count = 5` autoboxes an `int`; compare wrapper values with `.equals()` rather than `==`, which compares object references outside the guaranteed cached range. Validation cost per update is O(1), but public fields would allow callers to bypass checks and force duplicated validation throughout the application.

## Task 11 — Integrated implementation and validation
The integrated model supports multi-floor jagged slots, unique ticket IDs, duplicate-registration rejection, student/staff monthly passes, visitor billing, EV charging, entry/exit, and a daily collection total. The test run reports expected and actual outcomes.

In [ ]:
write_java("Task11", "SmartParkingSystem.java", """
import java.util.*;
import java.time.*;
abstract class Vehicle {
    private final String regNo;
    Vehicle(String regNo) {
        if (regNo == null || regNo.isBlank()) throw new IllegalArgumentException("Invalid registration");
        this.regNo = regNo;
    }
    public String getRegNo() { return regNo; }
    abstract double hourlyRate();
    double fee(long minutes) {
        if (minutes < 0) throw new IllegalArgumentException("Negative duration");
        return ((minutes + 59) / 60) * hourlyRate();
    }
}
interface Chargeable { double chargingFee(int kWh); }
class Bike extends Vehicle {
    Bike(String r) { super(r); }
    double hourlyRate() { return 10; }
}
class Sedan extends Vehicle {
    Sedan(String r) { super(r); }
    double hourlyRate() { return 30; }
}
class EVCar extends Sedan implements Chargeable {
    EVCar(String r) { super(r); }
    public double chargingFee(int kWh) { return kWh * 12.0; }
}
class ParkingTicket {
    final int id; final Vehicle vehicle; final int floor, slot; final long entryMinute;
    ParkingTicket(int id, Vehicle v, int f, int s, long t) {
        this.id=id; vehicle=v; floor=f; slot=s; entryMinute=t;
    }
}
class ParkingFacility {
    private final Vehicle[][] slots;
    private final Map<String, ParkingTicket> active = new LinkedHashMap<>();
    private static int nextTicket = 1001;
    private double collection = 0;
    ParkingFacility(int[] sizes) {
        slots = new Vehicle[sizes.length][];
        for (int i=0;i<sizes.length;i++) slots[i]=new Vehicle[sizes[i]];
    }
    ParkingTicket enter(Vehicle v, long minute, boolean validMonthlyPass) {
        if (active.containsKey(v.getRegNo())) throw new IllegalArgumentException("Duplicate registration");
        for (int f=0;f<slots.length;f++) for (int s=0;s<slots[f].length;s++) {
            if (slots[f][s]==null) {
                slots[f][s]=v;
                ParkingTicket t=new ParkingTicket(nextTicket++,v,f,s,minute);
                active.put(v.getRegNo(),t);
                return t;
            }
        }
        throw new IllegalStateException("Parking full");
    }
    double exit(String regNo, long exitMinute, boolean pass, int kWh) {
        ParkingTicket t=active.remove(regNo);
        if (t==null) throw new IllegalArgumentException("No active ticket");
        slots[t.floor][t.slot]=null;
        double amount=pass ? 0 : t.vehicle.fee(exitMinute-t.entryMinute);
        if (!pass && t.vehicle instanceof Chargeable c) amount += c.chargingFee(kWh);
        collection += amount;
        return amount;
    }
    double getCollection() { return collection; }
}
public class SmartParkingSystem {
    public static void main(String[] args) {
        ParkingFacility p=new ParkingFacility(new int[]{2,1});
        ParkingTicket a=p.enter(new Bike("TN01BI1111"),0,false);
        ParkingTicket b=p.enter(new Sedan("TN02CA2222"),0,false);
        ParkingTicket c=p.enter(new EVCar("TN03EV3333"),0,false);
        System.out.println("Tickets: "+a.id+", "+b.id+", "+c.id);
        System.out.println("Bike fee (60 min): Rs "+p.exit("TN01BI1111",60,false,0));
        System.out.println("Car fee (61 min): Rs "+p.exit("TN02CA2222",61,false,0));
        System.out.println("EV fee (60 min + 5 kWh): Rs "+p.exit("TN03EV3333",60,false,5));
        System.out.println("Daily collection: Rs "+p.getCollection());
        try { p.exit("UNKNOWN",70,false,0); }
        catch (IllegalArgumentException e) { System.out.println("Invalid exit handled: "+e.getMessage()); }
    }
}
""")
run_java("Task11", "SmartParkingSystem.java")

**Task 11 test evidence:** the executable scenario checks multiple vehicle types, fee boundary behaviour (60 and 61 minutes), EV charging, ticket uniqueness, and invalid exit. The test table in the report should include expected and actual output from this run. The active-ticket `HashMap` gives average O(1) lookup by registration; a linear array/list search is O(n). Both store O(n) active tickets.

## Task 12 — Day-4 design challenge
Changed requirements: e-bikes have a flat fee; delivery vans pay double the car rate from 9 AM to 5 PM; visitors can buy a one-day pass; five floors may have unequal slot counts. The implementation below uses a pricing abstraction and pass eligibility while retaining jagged floor storage.

In [ ]:
write_java("Task12", "DesignChallenge.java", """
import java.time.*;
interface PricingRule { double fee(long minutes, LocalTime entry, LocalTime exit); }
interface OneDayPass { boolean isValidFor(LocalDate date); }
class FlatEBikeRule implements PricingRule {
    public double fee(long minutes, LocalTime in, LocalTime out) { return 25.0; }
}
class DeliveryVanRule implements PricingRule {
    private final double carRate;
    DeliveryVanRule(double carRate) { this.carRate=carRate; }
    public double fee(long minutes, LocalTime in, LocalTime out) {
        long total=0;
        LocalTime t=in;
        for (long i=0;i<minutes;i++) {
            LocalTime current=t.plusMinutes(i);
            if (!current.isBefore(LocalTime.of(9,0)) && current.isBefore(LocalTime.of(17,0))) total++;
        }
        long regular=minutes-total;
        return ((total+59)/60.0)*(2*carRate) + ((regular+59)/60.0)*carRate;
    }
}
class VisitorDayPass implements OneDayPass {
    private final LocalDate validDate;
    VisitorDayPass(LocalDate date) { validDate=date; }
    public boolean isValidFor(LocalDate date) { return validDate.equals(date); }
}
public class DesignChallenge {
    public static void main(String[] args) {
        int[] slotsPerFloor={20,25,30,35,40};
        System.out.println("Floors: "+slotsPerFloor.length);
        System.out.println("E-bike flat fee: Rs "+new FlatEBikeRule().fee(180,LocalTime.NOON,LocalTime.of(15,0)));
        System.out.println("Delivery van fee (2h entirely in 9-5): Rs "+
            new DeliveryVanRule(30).fee(120,LocalTime.of(10,0),LocalTime.NOON));
        VisitorDayPass pass=new VisitorDayPass(LocalDate.of(2026,11,13));
        System.out.println("Visitor day pass valid: "+pass.isValidFor(LocalDate.of(2026,11,13)));
    }
}
""")
run_java("Task12", "DesignChallenge.java")

**Decision — MODIFY/partially redesign:** retain the tested jagged slot representation and common vehicle abstraction; introduce `PricingRule` so new pricing policies can be added without scattering conditions across vehicle classes; add a one-day pass validity object. Existing classes affected are the facility orchestration and pricing integration; new vehicle categories/rules are implemented as extensions. This reduces future changes to tested allocation code.

## 13. Collect all execution logs and create a ZIP
Run this final cell after every previous cell succeeds. It creates a README and a ZIP containing task folders, `.java` source files, compiled `.class` files (where generated), and execution logs. Download the ZIP from the Colab file browser or use the download command below.

In [ ]:
readme = """# CO1 Java Assignment 1 — Campus Smart-Parking System

Run environment: Google Colab, OpenJDK 17.
Run the notebook from top to bottom. Each Task folder contains its Java source and execution log.
Tasks 1, 2, 3–12 follow the uploaded assignment brief. The written report must include the analysis, pseudocode, UML, complexity comparisons, test table and design defence.

Compile/run locally:
    javac FileName.java
    java ClassName

Task 10 uses a package and is compiled with:
    javac -d . parking/model/ParkingSlot.java ParkingSlotApp.java
    java ParkingSlotApp
"""
(ROOT/"README.md").write_text(readme, encoding="utf-8")
zip_path = Path("/content/CSA09_Assignment1_Colab_Output.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for path in ROOT.rglob("*"):
        if path.is_file():
            z.write(path, path.relative_to(ROOT.parent))
print("ZIP created:", zip_path)
print("Files included:", sum(1 for p in ROOT.rglob("*") if p.is_file()))
from google.colab import files
files.download(str(zip_path))